<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Module_16_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

#Problem 01:

Iris Flower Classification using KNN
Dataset: Iris Dataset (UCI / Kaggle)
Dataset Link:
https://www.kaggle.com/datasets/uciml/iris

Tasks:
- Load the dataset using Pandas.
- Separate features and target.
- Split the data into training and testing sets.
- Train a KNN classifier with K=3.
- Predict the test set labels.
- Calculate the accuracy score.
- Display the Confusion Matrix.
- Generate the Classification Report.


In [41]:
# Load the dataset using Pandas
df_iris = pd.read_csv('Iris.csv')

# Separate features and target.
X = df_iris.drop(['Id', 'Species'], axis = 1)
y = df_iris['Species']

# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# Train a KNN classifier with K=3.
knn_pipeline = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=3, weights = 'uniform', algorithm = 'auto', p = 2, metric = 'minkowski'))
    ]
)
knn_pipeline.fit(X_train, y_train)

# Predict the test set labels.
y_pred = knn_pipeline.predict(X_test)

# Calculate the accuracy score.
accuracy = accuracy_score(y_test, y_pred)

# Display the Confusion Matrix.
conf_matrix = confusion_matrix(y_test, y_pred)
print('Confusion Matrix: \n', conf_matrix)

# Generate the Classification Report.
class_report = classification_report(y_test, y_pred)
print('Classification Report: \n', class_report)

Confusion Matrix: 
 [[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]
Classification Report: 
                  precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       0.83      1.00      0.91        10
 Iris-virginica       1.00      0.80      0.89        10

       accuracy                           0.93        30
      macro avg       0.94      0.93      0.93        30
   weighted avg       0.94      0.93      0.93        30



# Problem 02
Breast Cancer Diagnosis using Different K Values
Dataset: Breast Cancer Wisconsin Dataset
Dataset Link:
https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

Tasks:
- Load and preprocess the dataset.
- Standardize the features.
- Train KNN models for: K = 1, 3, 5, 7, 9, 11
- Record train and test accuracy.
- Determine the best K value.
- Explain the effect of very small and very large K.

In [42]:
# Load and preprocess the dataset.
df_cancer = pd.read_csv('data.csv')

# Standardize the features.
X = df_cancer.drop(['diagnosis', 'id', 'Unnamed: 32'], axis = 1)
y = df_cancer['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Train KNN models for: K = 1, 3, 5, 7, 9, 11
performance = []
acc = []

for k in range(1, 12, 2):
  knn_pipeline = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k, weights = 'uniform', algorithm = 'auto', p = 2, metric = 'minkowski'))
    ]
  )
  knn_pipeline.fit(X_train, y_train)

  y_pred = knn_pipeline.predict(X_test)

  # Record train and test accuracy.
  accuracy = accuracy_score(y_test, y_pred)
  acc.append(accuracy)

  # Determine the best K value.
  performance.append((k, accuracy))

print(performance)

# Explain the effect of very small and very large K.

# Small K (e.g. K=1):
# Model looks at only 1 nearest neighbor.
# Fits training data perfectly → high train accuracy
# But too sensitive to noise → low test accuracy (overfitting)

# Large K (e.g. K=11+):
# Model looks at many neighbors
# Decision becomes too generalized → underfitting
# May miss actual patterns in the data

[(1, 0.9385964912280702), (3, 0.9473684210526315), (5, 0.9473684210526315), (7, 0.9473684210526315), (9, 0.9649122807017544), (11, 0.956140350877193)]


# Problem 3
GridSearchCV for Customer Segmentation Prediction
Dataset: Customer Personality Analysis
Dataset Link:
https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis

Tasks:
- Create a customer response prediction problem using the Response column.
- Handle missing values.
- Scale the numerical features.
- Use GridSearchCV to tune:
   - n_neighbors
   - metric
   - weights
- Report the best parameters.
- Compare tuned KNN with default KNN.


In [45]:
# load dataset
df_customer = pd.read_csv('marketing_campaign.csv', sep='\t')

# Create a customer response prediction problem using the Response column.
df_customer.drop(columns=['ID', 'Dt_Customer', 'Z_CostContact', 'Z_Revenue'], inplace=True)

X = df_customer.drop('Response', axis = 1)
y = df_customer['Response']

cat_cols = ['Education', 'Marital_Status']
num_cols = [n for n in X.columns if n not in cat_cols]

# Handle missing values.
# Scale the numerical features.
num_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ]
)

# Handle missing values.
cat_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OneHotEncoder(sparse_output = False, handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)

knn_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('knn', KNeighborsClassifier())
    ]
)
knn_model.fit(X_train,y_train)

# Use GridSearchCV to tune: n_neighbors, metric, weights
grid = {
    'knn__n_neighbors' : range(1, 12, 2),
    'knn__metric': ['euclidean', 'manhattan', 'minkowski'],
    'knn__weights': ['uniform', 'distance']
}

grid_search = GridSearchCV(
    estimator = knn_model,
    param_grid = grid,
    cv = 5
)
grid_search.fit(X_train, y_train)

# Report the best parameters.
best_param = grid_search.best_params_
print('Best Parameters:', best_param)

# Compare tuned KNN with default KNN.
grid_pred = grid_search.predict(X_test)
tuned_acc = accuracy_score(y_test, grid_pred)
print(f"Tuned KNN Accuracy: {tuned_acc:.4f}")

# Comparison
knn_pred = knn_model.predict(X_test)
default_acc = accuracy_score(y_test, knn_pred)
print(f"\nDefault KNN: {default_acc:.4f}")
print(f"Tuned KNN:   {tuned_acc:.4f}")

Best Parameters: {'knn__metric': 'euclidean', 'knn__n_neighbors': 11, 'knn__weights': 'distance'}
Tuned KNN Accuracy: 0.8906

Default KNN: 0.8728
Tuned KNN:   0.8906
